# Four-method measurement-aware regression walkthrough

This notebook demonstrates completed code and preserved results. It does not tune methods on test outcomes or claim that covariance regularization improves every setting.

In [ ]:
from pathlib import Path
import sys, json
import numpy as np
ROOT=Path.cwd()
if not (ROOT/"src").exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/"src"))
from measurement_extension.inference import load_readout
from measurement_extension.covariance import covariance_of_means
DIRECTORY=ROOT/"results/measurement_extension"
config=json.loads((DIRECTORY/"config.json").read_text())
print("Datasets:",len(config["data_seeds"]),"Methods:",config["methods"])


## Reconstruct one measurement-noise covariance

Use only training counts. The matrix describes uncertainty at fixed inputs, rather than variation of features across inputs. It has three independent setting blocks.

In [ ]:
ds=config["data_seeds"][0]; ms=config["measurement_seeds"][0]; shots=config["shots_per_setting"][0]
with np.load(DIRECTORY/"raw"/f"records_d{ds}_m{ms}_s{shots}.npz") as a:
    training_counts=a["counts"][:config["n_train"]].copy()
covariance=covariance_of_means(training_counts).mean(axis=0)
print("Covariance shape:",covariance.shape)
print("Smallest eigenvalue:",np.linalg.eigvalsh(covariance).min())
print("Within-setting first block:\n",covariance[:3,:3])


## Load all four trained readouts

No retraining or model selection occurs here. Each model is identified by the same dataset, measurement acquisition and shot budget.

In [ ]:
models={m:load_readout(DIRECTORY,method=m,task="sine",shots=shots) for m in config["methods"]}
for name,model in models.items():
    r=model.record
    print(name,"lambda=",r["lambda_"],"gamma=",r["gamma"],"recorded test MSE=",r["test_mse"])


## Predict at new inputs with common simulated measurements

The same sampling seed and shot count give each readout the same new feature measurements. This is a demonstration, not an aggregate performance benchmark.

In [ ]:
inputs=np.array([-0.5,0.2,0.9])
outputs={m:model.predict_inputs(inputs,sampling_seed=9001) for m,model in models.items()}
for name,output in outputs.items():
    np.testing.assert_array_equal(output["counts"],outputs["OLS"]["counts"])
    print(name,output["predictions"])
print("Common record costs",outputs["OLS"]["simulated_shots"],"simulated executions if acquired once and reused.")


## Read the full fresh-seed comparison

Aggregate acquisitions within datasets first. The full covariance method improves only one of six primary mean errors versus Ridge in this study. These means do not establish significance.

In [ ]:
summary=json.loads((DIRECTORY/"summary.json").read_text())
for row in summary["contrasts"]:
    if row["label_noise_std"]==0 and row["shots"]>0 and row["reference"]=="Ridge" and row["candidate"]=="FullCov":
        print(row["task"],row["shots"],f"{row["relative_mse_reduction_percent"]:+.2f}% reduction",row["dataset_means_favouring_candidate"],"of",row["n_data_seeds"],"dataset means favour FullCov")


## Refit and verify from the command line

Run `python scripts/defense_demo.py --refit` for a one-case retraining demonstration, or `python scripts/verify_project.py` for all tests, both independent audits and fresh reproductions. Canonical experimental results are preserved. The original baseline notebook remains available separately.